# 07 - SQLAlchemy e PostgreSQL

Conexao com PostgreSQL usando SQLAlchemy e Pandas.

## 1. Conexao

In [ ]:
import os
import sqlalchemy

user = os.environ.get('POSTGRES_USER', 'elt')
passw = os.environ.get('POSTGRES_PASSWORD')
if not passw:
    raise RuntimeError('POSTGRES_PASSWORD nao definida no ambiente. Configure o .env e reinicie o container.')
host = 'elt-postgres'
port = 5432
dbname = 'postgres'

url = f'postgresql+psycopg2://{user}:{passw}@{host}:{port}/{dbname}'
masked = url.replace(passw, '***')
print(f'URL: {masked}')

engine = sqlalchemy.create_engine(url)
with engine.connect() as conn:
    r = conn.execute(sqlalchemy.text('SELECT 1'))
    print(f'Conexao OK: {r.scalar()}')

## 2. Tabelas Existentes

In [ ]:
import pandas as pd

tables = pd.read_sql("""
    SELECT schemaname, tablename
    FROM pg_tables
    WHERE schemaname NOT IN ('pg_catalog', 'information_schema')
    ORDER BY schemaname, tablename
""", engine)
print(tables)

## 3. Criar Tabela e Inserir

In [ ]:
# O schema "global" e a convencao do projeto (.env: *_SCHEMA=global)
with engine.begin() as conn:
    conn.execute(sqlalchemy.text('CREATE SCHEMA IF NOT EXISTS global'))

with engine.begin() as conn:
    conn.execute(sqlalchemy.text("""
        CREATE TABLE IF NOT EXISTS global.datalab_teste (
            id SERIAL PRIMARY KEY,
            nome VARCHAR(100),
            salario NUMERIC(10,2)
        )
    """))

with engine.begin() as conn:
    conn.execute(sqlalchemy.text("DELETE FROM global.datalab_teste"))
    conn.execute(sqlalchemy.text("""
        INSERT INTO global.datalab_teste (nome, salario)
        VALUES ('Ana', 9500), ('Joao', 7200), ('Maria', 11000)
    """))
print('Dados inseridos')

## 4. SELECT com Pandas

In [ ]:
df = pd.read_sql('SELECT * FROM global.datalab_teste', engine)
print(df)

## 5. Cleanup

In [ ]:
with engine.begin() as conn:
    conn.execute(sqlalchemy.text('DROP TABLE IF EXISTS global.datalab_teste'))
print('Tabela removida')

## 6. Exercicio

Crie uma tabela `global.datalab_funcionarios` com id, nome, depto e salario. Insira 5 registros e faca um SELECT com WHERE.

## Conclusao

Nunca exiba credenciais. Use variaveis de ambiente. Sempre faca cleanup de tabelas de laboratorio.